In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from IPython.display import display, Markdown

In [3]:
# 데이터 전처리

# 데이터 로드
file_path = r'C:\Users\esang\Documents\work\devel\kdy\data\raw\kpf2025.csv'
df = pd.read_csv(file_path)
print(df.head())

   caseid  region  sex  age  hhsize  hhtype  educ  occupation  income  class  \
0       2      17    1   57       4       2     3           3       4      3   
1       3      17    1   71       2       1     2           4       2      3   
2       7      17    2   82       1       1     2          10       2      3   
3       9      17    2   57       6       3     2           3       3      1   
4      11      17    1   55       3       2     3           5       3      3   

   ...  willing_pay  civic_1  civic_2  civic_3  civic_4  civic_5  press_eval  \
0  ...            2        2        2        2        2        2         3.0   
1  ...            2        2        2        2        2        2         3.6   
2  ...            2        2        2        2        2        2         3.6   
3  ...            2        2        2        2        2        2         2.8   
4  ...            2        2        2        2        2        2         3.0   

   press_role  press_problem  journali

### kpf2025 '인터넷/포털 뉴스 이용(포털 중심 미디어 소비)'을 핵심 주제로 잡고, H1(독립표본 t-검정)과 H2(카이제곱 검정) 선정

1. H1: 독립표본 t-검정 (Independent-samples t test)
    - 가설: 평소 포털 뉴스를 이용하는 집단(`use_portalnews`=1)은 이용하지 않는 집단(`use_portalnews`=2)보다 언론 전반에 대한 신뢰 평가 점수(`press_eval`)가 더 낮을 것이다.
    - 이유: 전통 매체(종이신문/TV) 위주로 뉴스를 소비하는 사람보다 포털/인터넷 환경에서 알고리즘과 댓글에 노출되는 사람들이 언론의 공정성이나 신뢰도를 비판적으로 평가하는 경향이 있는지 확인한다. 독립변수(포털 이용 여부)가 2집단이고, 종속변수(신뢰도 평균 점수)가 연속형이므로 t-검정에 부합한다.

2. H2: 카이제곱 독립성 검정 (Chi-square test of independence)
    - 가설: 연령대(청년층 vs 중장년층 vs 노년층)에 따라 주이용 뉴스 매체가 TV 뉴스(`use_tvnews`)인지 동영상 뉴스(`use_videonews` - 예: 유튜브)인지 그 선택 비율에 차이가 있을 것이다.
    - 이유: 연령대(범주형)와 특정 뉴스 매체 이용 여부(범주형) 간의 연관성을 분할표로 확인하는 분석이다. 세대별로 선호하는 매체가 다르다는 것을 증명할 수 있다.

## H1. 독립표본 t-검정 (포털뉴스 이용 여부 -> 언론 평가 점수)

In [5]:
# 결측치 처리 규칙 적용 (-1, 998, 999) -> NaN
df.replace([-1, 998, 999], np.nan, inplace=True)

# H2 분석을 위해 연령(age)을 3개 범주로 묶기
def categorize_age(age):
    if age < 40: return '청년층(2030)'
    elif age < 60: return '중장년층(4050)'
    else: return '노년층(60+)'
df['age_group'] = df['age'].apply(categorize_age)

## [H1] 포털 뉴스 이용 여부에 따른 언론 전반 평가 점수 차이 (t-test)

In [7]:
display(Markdown("### [H1] 포털 뉴스 이용 여부에 따른 언론 전반 평가 점수 차이 (t-test)"))

h1_df = df.dropna(subset=['use_portalnews', 'press_eval']).copy()
# use_portalnews: 1=이용, 2=이용 안함
group_use = h1_df[h1_df['use_portalnews'] == 1]['press_eval']
group_not = h1_df[h1_df['use_portalnews'] == 2]['press_eval']

# 1. Levene 등분산 검정

# 비교하려는 두 집단의 데이터가 퍼져있는 정도(분산)가 비슷한지 확인하는 사전 검사
# t-검정이나 분산분석(anova)은 기본적으로 "집단들의 데이터 퍼짐 정도가 비슷하다"는 전제 조건이 필요하다.
# levene 검정을 통해 이 조건이 맞는지 먼저 확인 후 분산이 비슷하다면 일반적인 방식을,
# 분산이 다르다면 수식이 보정된 방식(welch's test)을 선택하여 분석의 정확도를 높이기 위해 사용한다. (pdf에 있음)

levene_stat, levene_p = stats.levene(group_use, group_not)
equal_var = levene_p >= 0.05

# 2. t-검정 (등분산 가정 충족 여부에 따라 자동으로 방식 결정)
t_stat, p_val = stats.ttest_ind(group_use, group_not, equal_var=equal_var)

# 3. Cohen's d (효과 크기) 계산

# 두 집단의 평균 차이가 실질적으로 얼마나 '큰' 차이인지를 보여주는 지표
# t-검정 결과(p-value)는 두 집단 간에 차이가 우연인지 아닌지만 알려준다.
# cohen's d를 통해 차이가 현실적으로 의미가 있는지를 객관적인 수치로 확인하기 위해 사용한다. (pdf에 있음)

n1, n2 = len(group_use), len(group_not)
var1, var2 = group_use.var(), group_not.var()
pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
cohens_d = (group_use.mean() - group_not.mean()) / pooled_std

t_test_results = pd.DataFrame({
    '집단': ['포털뉴스 이용', '포털뉴스 비이용'],
    '표본 수(N)': [n1, n2],
    '평균 점수': [round(group_use.mean(), 2), round(group_not.mean(), 2)],
    '표준편차': [round(group_use.std(), 2), round(group_not.std(), 2)],
    't-통계량': [round(t_stat, 4), '-'],
    'p-value': [f"{p_val:.4f}", '-'],
    "Cohen's d": [round(cohens_d, 4), '-']
})

display(t_test_results)

### [H1] 포털 뉴스 이용 여부에 따른 언론 전반 평가 점수 차이 (t-test)

,집단,표본 수(N),평균 점수,표준편차,t-통계량,p-value,Cohen's d
0,포털뉴스 이용,1802,3.48,0.58,0.9222,0.3566,0.1156
1,포털뉴스 비이용,66,3.41,0.64,-,-,-


### H1: 포털 뉴스 이용 여부에 따른 언론 전반 평가 점수 차이 (독립표본 t-검정)

1. 결과 요약
    * 평균 차이: 포털 뉴스를 이용하는 집단의 언론 신뢰 평가 점수(3.48점)가 이용하지 않는 집단(3.41점)보다 근소하게 높게 나타났다.
    * 통계적 유의성 (p=0.3566): p값이 기준치(0.05)보다 크므로, 두 집단 간의 점수 차이는 통계적으로 유의미하지 않다. (가설 기각)
    * 효과 크기 (Cohen's d = 0.1156): 효과 크기 역시 매우 작아 두 집단 간의 실질적인 차이를 발견하기 어렵다.

2. 해석
    * 포털 뉴스 이용 여부에 따라 언론 전반을 평가하는 신뢰도 점수에는 통계적으로 유의미한 차이가 없는 것으로 확인되었다.
    * 이는 뉴스 소비 환경(포털 vs 전통 매체)이 변하더라도, 한국 언론 전반에 대해 수용자들이 느끼는 공정성이나 평가는 큰 차이 없이 비슷한 수준을 형성하고 있음을 시사한다.
---

## H2. 카이제곱 독립성 검정 (연령대 vs 동영상(유튜브) 뉴스 이용 여부)

In [8]:
display(Markdown("### [H2] 연령대와 동영상 플랫폼(유튜브 등) 뉴스 이용 여부의 연관성 (Chi-square)"))

# 분석 변수 결측치 제거
h2_df = df.dropna(subset=['age_group', 'use_videonews']).copy()
# use_videonews: 1=이용, 2=비이용. 가독성을 위해 텍스트로 변환
h2_df['video_news'] = h2_df['use_videonews'].map({1: '이용', 2: '이용안함'})

# 교차표 생성
contingency_table = pd.crosstab(h2_df['age_group'], h2_df['video_news'])

# 카이제곱 검정
chi2, p_val_chi, dof, expected = stats.chi2_contingency(contingency_table)

# Cramer's V (효과 크기) 계산
n_total = contingency_table.sum().sum()
k = min(contingency_table.shape)
cramers_v = np.sqrt(chi2 / (n_total * (k - 1)))

chi2_results = pd.DataFrame({
    '분석 내용': ['연령대 vs 동영상 뉴스 이용'],
    '총 표본 수(N)': [n_total],
    '최소 기대빈도': [round(expected.min(), 2)],
    '카이제곱 통계량': [round(chi2, 4)],
    '자유도(df)': [dof],
    'p-value': [f"{p_val_chi:.4f}"],
    "Cramer's V": [round(cramers_v, 4)]
})

display(chi2_results)

display(Markdown("**▼ 연령대별 동영상 뉴스 이용 빈도 (교차표)**"))
display(contingency_table)

### [H2] 연령대와 동영상 플랫폼(유튜브 등) 뉴스 이용 여부의 연관성 (Chi-square)

,분석 내용,총 표본 수(N),최소 기대빈도,카이제곱 통계량,자유도(df),p-value,Cramer's V
0,연령대 vs 동영상 뉴스 이용,1868,136.49,53.9357,2,0.0000,0.1699


**▼ 연령대별 동영상 뉴스 이용 빈도 (교차표)**

video_news,이용,이용안함
age_group,,
노년층(60+),106,540
중장년층(4050),218,491
청년층(2030),173,340


### H2: 연령대와 동영상(유튜브) 뉴스 이용 여부의 연관성 (카이제곱 독립성 검정)

1. 결과 요약
    * 통계적 유의성 (p=0.0000): p값이 0.05보다 현저히 작으므로, 연령대와 동영상 뉴스 이용 여부 사이에는 강력하고 유의미한 연관성이 존재한다.
    * 효과 크기 (Cramer's V = 0.1699): 두 변수 간의 연관성 강도는 약 0.17 수준으로 '작은~중간 정도의 효과'를 가지는 것으로 해석한다.
    * 빈도 분포(비율): 청년층(약 33.7%)과 중장년층(약 30.7%)은 동영상 뉴스 이용 비율이 상대적으로 높은 반면, 노년층(약 16.4%)은 비이용자의 비율이 압도적으로 높았다.

2. 해석
    * 연령대에 따라 동영상 플랫폼(유튜브 등)을 통한 뉴스 소비 방식에 통계적으로 유의미한 차이가 있음이 확인되었다.
    * 특히 청년 및 중장년층에서는 3명 중 1명 꼴로 동영상 뉴스를 적극적으로 소비하고 있으나, 노년층에서는 그 비율이 절반 수준으로 급감한다. 이를 통해 연령이 낮을수록 텍스트 기반이나 전통 TV를 넘어 동영상 플랫폼을 뉴스 창구로 적극 활용하는 세대별 미디어 분화 현상을 알 수 있다.

## 결과 종합

1. 분석 결과의 종합적 의미
* 이번 분석을 통해 미디어 수용자들의 뉴스 소비 방식이 세대별로 분화되고 있으나, 매체 이용 방식이 언론에 대한 본질적인 인식 격차로 바로 이어지지는 않는다는 점을 확인했다.
* H2의 결과처럼, 청년 및 중장년층을 중심으로 유튜브 등 동영상 플랫폼을 통한 뉴스 소비가 활발해지며 미디어 이용의 세대 간 다변화가 뚜렷하게 나타나고 있다. 하지만 결과에서 보듯이 포털 뉴스를 이용하든 전통 매체를 고수하든 언론 전반에 대해 부여하는 평가 점수에는 유의미한 차이가 없었다.
* 즉, 사람들은 자신의 연령이나 라이프스타일에 맞춰 '뉴스를 소비하는 플랫폼'은 다르게 선택하고 있지만 언론의 공정성이나 신뢰도라는 '본질적 콘텐츠'에 대해서는 매체 이용 환경과 무관하게 비슷한 수준의 비판적, 혹은 중도적 시각을 공유하고 있음을 시사한다.

2. 분석의 한계점
* 첫째, 본 분석은 특정 매체(포털, 동영상)의 '이용 여부'에 초점을 맞추었을 뿐, 수용자들이 해당 매체에서 구체적으로 어떤 주제의 뉴스를 소비하고(예: 정치, 연예, 경제), 뉴스를 얼마나 깊이 있게 읽는지(이용 시간 및 관여도)는 반영하지 못했다.
* 둘째, 연령이 낮을수록 동영상 뉴스 이용 빈도가 높다는 상관관계는 확인했으나, 노년층이 동영상 뉴스를 덜 보는 이유가 단순한 '디지털 기기 조작의 어려움' 때문인지, 아니면 '전통 TV 매체에 대한 강한 신뢰와 선호' 때문인지 그 인과적 이유는 이 데이터만으로는 명확히 알 수 없다.
* 셋째, 이 데이터는 2025년의 단면적 설문조사이므로, 세대별 뉴스 소비 패턴이 과거에 비해 어떻게 변화해 왔는지는 추가적인 시계열 데이터 분석이 필요하다.